In [ ]:
import sys
sys.path.insert(0, '/content/lung-nodule-fusion-xai')
import numpy as np
import torch


In [ ]:
# Grad-CAM demo with random weights (replace with loaded checkpoint for real analysis)
from src.models.backbones import BackboneClassifier
from src.xai.gradcam_utils import compute_gradcam, overlay_cam_on_image

model = BackboneClassifier('mobilenet_v3_large', n_input_channels=3, pretrained=False)
model.eval()

img_tensor = torch.randn(1, 3, 64, 64)
cam_map = compute_gradcam(model, img_tensor, backbone_name='mobilenet_v3_large',
                           target_class=1, method='gradcam')
print(f"CAM map shape: {cam_map.shape}, range: [{cam_map.min():.3f}, {cam_map.max():.3f}]")


In [ ]:
# TreeSHAP demo on synthetic radiomic data
from src.xai.shap_utils import compute_tree_shap, get_top_shap_features
import numpy as np
import xgboost as xgb

np.random.seed(42)
n, f = 50, 30
feature_names = [f'original_glcm_feat_{i}' for i in range(15)] +                 [f'original_shape_feat_{i}' for i in range(15)]
X = np.random.randn(n, f)
y = (X[:, 0] + X[:, 1] > 0).astype(int)

clf = xgb.XGBClassifier(n_estimators=50, random_state=42)
clf.fit(X, y)

shap_vals, explainer = compute_tree_shap(clf, X, feature_names, output_dir='/content/xai_output')
top_df = get_top_shap_features(shap_vals, feature_names, n_top=5)
print("Top SHAP features:")
print(top_df)


In [ ]:
# Spatial cross-check (Grad-CAM location vs nodule mask)
from src.xai.shap_utils import spatial_cross_check

# Simulate 10 test nodules
cam_maps = [np.random.rand(64, 64) for _ in range(10)]
nodule_masks = [np.random.randint(0, 2, (64, 64)).astype(bool) for _ in range(10)]

result = spatial_cross_check(shap_vals, feature_names, cam_maps, nodule_masks)
print(f"Texture dominated: {result['texture_dominated']}")
print(f"Mean CAM in nodule: {result['mean_cam_in_nodule']:.3f}")
print(f"Spurious activation flag: {result['spurious_flag']}")
